# Learning predictions for downstream decisions

## Separating prediction error from decision regret

Let the feasible decisions be the rows of a supplied matrix $V$. For a cost vector $c$, an exact finite oracle returns

$$
w^{*}(c)=\arg\min_{w\in V} c^{\mathsf T}w.
$$

Here, $V$ is the finite list of feasible vertices, $c$ contains the costs, and $w^{*}(c)$ is the first listed decision with the smallest linear cost. The implemented oracle is exact for the convex hull of these supplied vertices because a linear objective reaches its minimum at a vertex. It does not create a feasible region or enumerate missing vertices.

### Regret from a Predicted Cost

If $\widehat c$ is a predicted cost vector, the decision loss is

$$
L_{\mathrm{SPO}}(\widehat c,c)
=
c^{\mathsf T}w^{*}(\widehat c)
-
c^{\mathsf T}w^{*}(c).
$$

The vector $c$ is the observed cost, $\widehat c$ is its prediction, and the two oracle calls select the predicted and observed best decisions. The loss is the extra observed cost caused by acting on the prediction. It answers a different question from a squared prediction error.

In [1]:
from io import StringIO
from itertools import combinations
from pathlib import Path
from zipfile import ZipFile
import sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display

def display(value):
    module = value.__class__.__module__
    if module.startswith('pandas') and hasattr(value, '_repr_html_'):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + '</div>'))
    return _display(value)
BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset
import RiskLabAI
from RiskLabAI.optimization import robust_mean_variance, spo_plus_loss, spo_plus_torch
assert RiskLabAI.__version__ == '3.2.0'
DATA_PATH = ensure_dataset(BOOK_ROOT, 'french_49_industries')
FIGURES = BOOK_ROOT / 'outputs' / 'figures' / '27_decision_focused_learning'
TABLES = BOOK_ROOT / 'outputs' / 'tables' / '27_decision_focused_learning'
FIGURE_DATA = BOOK_ROOT / 'outputs' / 'figure_data' / '27_decision_focused_learning'
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)
PALETTE = {'background': '#111827', 'axes': '#111827', 'text': '#E5E7EB', 'cyan': '#38BDF8', 'amber': '#FBBF24', 'green': '#34D399', 'rose': '#FB7185', 'violet': '#C084FC', 'grid': '#374151'}
plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 11.5, 'figure.facecolor': PALETTE['background'], 'axes.facecolor': PALETTE['axes'], 'axes.edgecolor': PALETTE['text'], 'axes.labelcolor': PALETTE['text'], 'axes.titlecolor': PALETTE['text'], 'xtick.color': PALETTE['text'], 'ytick.color': PALETTE['text'], 'text.color': PALETTE['text'], 'grid.color': PALETTE['grid'], 'grid.linestyle': ':', 'legend.facecolor': PALETTE['axes'], 'legend.edgecolor': PALETTE['grid'], 'savefig.facecolor': PALETTE['background'], 'savefig.transparent': False, 'savefig.dpi': 300})

def rendered_value(value):
    if pd.isna(value):
        return ''
    if isinstance(value, (bool, np.bool_)):
        return 'Pass' if bool(value) else 'Fail'
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    if isinstance(value, (int, np.integer)):
        return f'{int(value):,}'
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if number == 0:
            return '0'
        if number.is_integer():
            return f'{int(number):,}'
        magnitude = abs(number)
        if magnitude < 0.0001 or magnitude >= 1000000:
            return f'{number:.6g}'
        return f'{number:.6f}'.rstrip('0').rstrip('.')
    return str(value)

def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))

def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f'{stem}.csv', index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position='htbp')
    if fit:
        latex = latex.replace('\\begin{tabular}', '\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}', 1).replace('\\end{tabular}', '\\end{tabular}\\end{adjustbox}', 1)
    (TABLES / f'{stem}.tex').write_text(latex, encoding='utf-8')
    display(shown)
    return shown

def finish_axis(axis, grid=True):
    axis.spines['top'].set_visible(False)
    axis.spines['right'].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)

def save_figure(figure, stem):
    figure.savefig(FIGURES / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    figure.savefig(FIGURES / f'{stem}.pdf', bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    display(figure)

def load_public_returns():
    assert DATA_PATH.is_file()
    with ZipFile(DATA_PATH) as archive:
        member = archive.namelist()[0]
        lines = archive.read(member).decode('latin1').splitlines()
    title_index = next((index for index, line in enumerate(lines) if 'Average Value Weighted Returns -- Daily' in line))
    header_index = title_index + 1
    end_index = next((index for index in range(header_index + 1, len(lines)) if not lines[index].strip()))
    frame = pd.read_csv(StringIO('\n'.join(lines[header_index:end_index])))
    frame = frame.rename(columns={frame.columns[0]: 'Date'})
    frame['Date'] = pd.to_datetime(frame['Date'].astype(str), format='%Y%m%d')
    industries = list(frame.columns[1:17])
    frame[industries] = frame[industries].apply(pd.to_numeric, errors='coerce')
    frame[industries] = frame[industries].mask(frame[industries] <= -99)
    frame = frame.loc[(frame['Date'] >= '2019-01-01') & (frame['Date'] <= '2025-12-31'), ['Date', *industries]].dropna().reset_index(drop=True)
    assert len(industries) == 16
    assert len(frame) == 1760
    return (frame, industries)
import torch
vertices = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [0.2, 0.8]])
true_cost = np.array([2.0, -1.0])
predicted_cost = np.array([-0.4, 0.8])
result = spo_plus_loss(predicted_cost, true_cost, vertices)
notation = pd.DataFrame([{'Symbol': 'V', 'Description': 'Matrix whose rows are the supplied feasible vertices'}, {'Symbol': 'c', 'Description': 'Observed linear cost vector'}, {'Symbol': 'c hat', 'Description': 'Predicted linear cost vector'}, {'Symbol': 'w*(c)', 'Description': 'First supplied vertex with the lowest observed cost'}, {'Symbol': 'L_SPO', 'Description': 'Extra observed cost caused by the predicted decision'}, {'Symbol': 'L_SPO+', 'Description': 'Convex upper surrogate returned by RiskLabAI'}])
save_table(notation, 'decision_loss_notation', 'Notation for decision loss and the SPO+ surrogate.', 'tab:chapter28-decision-loss-notation', fit=True)
vertex_table = pd.DataFrame(vertices, columns=['Decision component 1', 'Decision component 2'])
vertex_table.insert(0, 'Vertex', np.arange(len(vertices)))
save_table(vertex_table, 'finite_feasible_vertices', 'Explicitly supplied feasible vertices in the exact example.', 'tab:chapter28-finite-vertices')
true_objectives = vertices @ true_cost
predicted_objectives = vertices @ predicted_cost
transformed_objectives = vertices @ (2 * predicted_cost - true_cost)
oracle_table = pd.DataFrame({'Vertex': np.arange(len(vertices)), 'Observed objective': true_objectives, 'Predicted objective': predicted_objectives, 'Transformed objective': transformed_objectives, 'Observed oracle': np.arange(len(vertices)) == np.argmin(true_objectives), 'Predicted oracle': np.arange(len(vertices)) == np.argmin(predicted_objectives), 'Transformed oracle': np.arange(len(vertices)) == np.argmin(transformed_objectives)})
save_table(oracle_table, 'finite_oracle_checks', 'Exhaustive objective values and selected vertices.', 'tab:chapter28-finite-oracle-checks', fit=True)
decision_regret = float(true_cost @ vertices[np.argmin(predicted_objectives)] - true_objectives.min())
manual_loss = float((true_cost - 2 * predicted_cost) @ vertices[np.argmin(transformed_objectives)] + (2 * predicted_cost - true_cost) @ vertices[np.argmin(true_objectives)])
loss_checks = pd.DataFrame([{'Quantity': 'Observed best vertex', 'Value': np.argmin(true_objectives), 'Check': True}, {'Quantity': 'Predicted selected vertex', 'Value': np.argmin(predicted_objectives), 'Check': True}, {'Quantity': 'Decision regret', 'Value': decision_regret, 'Check': decision_regret >= 0}, {'Quantity': 'Manual SPO+ loss', 'Value': manual_loss, 'Check': np.isclose(manual_loss, result['loss'])}, {'Quantity': 'RiskLabAI SPO+ loss', 'Value': result['loss'], 'Check': result['loss'] >= decision_regret - 1e-12}])
save_table(loss_checks, 'spo_loss_components', 'Decision regret and exhaustive SPO+ checks.', 'tab:chapter28-spo-loss-components')
plot_data = pd.DataFrame({'Decision': [f'Vertex {index}' for index in range(len(vertices))], 'Observed cost': true_objectives, 'Predicted cost': predicted_objectives})
plot_data.to_csv(FIGURE_DATA / 'exact_cost_decisions.csv', index=False)
figure, axis = plt.subplots(figsize=(9.2, 5.3), constrained_layout=True)
x = np.arange(len(vertices))
axis.bar(x - 0.19, true_objectives, width=0.38, color=PALETTE['green'], label='Observed objective')
axis.bar(x + 0.19, predicted_objectives, width=0.38, color=PALETTE['amber'], label='Predicted objective')
axis.set_xticks(x, plot_data['Decision'])
axis.set_ylabel('Linear cost')
axis.set_title('Predicted and Observed Costs Can Select Different Decisions')
axis.legend()
finish_axis(axis)
save_figure(figure, 'exact_cost_decisions')
counter_true = np.array([0.0, 0.01, 10.0])
predictions = {'Lower error, wrong decision': np.array([0.006, 0.004, 10.0]), 'Higher error, right decision': np.array([0.0, 0.1, 9.5])}
counter_vertices = np.eye(3)
counter_rows = []
for name, prediction in predictions.items():
    selected = int(np.argmin(counter_vertices @ prediction))
    counter_rows.append({'Prediction': name, 'Mean squared error': float(np.mean((prediction - counter_true) ** 2)), 'Selected vertex': selected, 'Decision regret': float(counter_true[selected] - counter_true.min())})
counter_table = pd.DataFrame(counter_rows)
assert counter_table.loc[0, 'Mean squared error'] < counter_table.loc[1, 'Mean squared error']
assert counter_table.loc[0, 'Decision regret'] > counter_table.loc[1, 'Decision regret']
save_table(counter_table, 'prediction_error_decision_error', 'A lower prediction error need not produce a lower decision regret.', 'tab:chapter28-error-regret-counterexample', fit=True)
counter_table.to_csv(FIGURE_DATA / 'mse_regret_contrast.csv', index=False)
figure, axes = plt.subplots(1, 2, figsize=(10.4, 4.8), constrained_layout=True)
axes[0].bar(counter_table['Prediction'], counter_table['Mean squared error'], color=[PALETTE['rose'], PALETTE['cyan']])
axes[1].bar(counter_table['Prediction'], counter_table['Decision regret'], color=[PALETTE['rose'], PALETTE['cyan']])
axes[0].set_title('Prediction Error')
axes[1].set_title('Decision Regret')
axes[0].set_ylabel('Mean squared error')
axes[1].set_ylabel('Extra observed cost')
for axis in axes:
    axis.tick_params(axis='x', rotation=18)
    finish_axis(axis)
save_figure(figure, 'mse_regret_contrast')


,Symbol,Description
0,V,Matrix whose rows are the supplied feasible ve...
1,c,Observed linear cost vector
2,c hat,Predicted linear cost vector
3,w*(c),First supplied vertex with the lowest observed...
4,L_SPO,Extra observed cost caused by the predicted de...
5,L_SPO+,Convex upper surrogate returned by RiskLabAI


,Vertex,Decision component 1,Decision component 2
0,0,0,0
1,1,1,0
2,2,0,1
3,3,0.2,0.8


,Vertex,Observed objective,Predicted objective,Transformed objective,Observed oracle,Predicted oracle,Transformed oracle
0,0,0,0,0,Fail,Fail,Fail
1,1,2,-0.4,-2.8,Fail,Pass,Pass
2,2,-1,0.8,2.6,Pass,Fail,Fail
3,3,-0.4,0.56,1.52,Fail,Fail,Fail


,Quantity,Value,Check
0,Observed best vertex,2,Pass
1,Predicted selected vertex,1,Pass
2,Decision regret,3,Pass
3,Manual SPO+ loss,5.4,Pass
4,RiskLabAI SPO+ loss,5.4,Pass


<Figure size 920x530 with 1 Axes>

,Prediction,Mean squared error,Selected vertex,Decision regret
0,"Lower error, wrong decision",2.4e-05,1,0.01
1,"Higher error, right decision",0.086033,0,0


<Figure size 1040x480 with 2 Axes>

## The SPO+ Surrogate

The finite SPO+ loss used here is

$$
L_{\mathrm{SPO+}}(\widehat c,c)
=
(c-2\widehat c)^{\mathsf T}w^{*}(2\widehat c-c)
+
(2\widehat c-c)^{\mathsf T}w^{*}(c).
$$

The transformed oracle uses the cost $2\widehat c-c$. Under the stated linear decision setting, this loss is a convex upper surrogate for decision loss. One selected subgradient with respect to $\widehat c$ is

$$
g(\widehat c,c)
=
2\left[w^{*}(c)-w^{*}(2\widehat c-c)\right].
$$

The vector $g(\widehat c,c)$ gives a valid supporting slope at the supplied prediction. At ties, the released implementation selects the first supplied vertex, so the returned subgradient is deterministic but need not be unique.

In [2]:
prediction_tensor = torch.tensor(predicted_cost, dtype=torch.float64, requires_grad=True)
truth_tensor = torch.tensor(true_cost, dtype=torch.float64)
vertex_tensor = torch.tensor(vertices, dtype=torch.float64)
torch_loss = spo_plus_torch(prediction_tensor, truth_tensor, vertex_tensor)
torch_loss.backward()
gradient_table = pd.DataFrame({
    "Component": [1, 2],
    "RiskLabAI subgradient": result["subgradient"],
    "Torch gradient": prediction_tensor.grad.detach().numpy(),
    "Absolute difference": np.abs(result["subgradient"] - prediction_tensor.grad.detach().numpy()),
})
assert np.allclose(gradient_table["RiskLabAI subgradient"], gradient_table["Torch gradient"])
save_table(gradient_table, "spo_torch_gradient_checks", "Selected SPO+ subgradient and tensor gradient.", "tab:chapter28-spo-gradient-checks")

rng = np.random.default_rng(2801)
shifted = rng.normal(size=(100, 2))
margins = []
for index, point in enumerate(shifted):
    shifted_loss = spo_plus_loss(point, true_cost, vertices)["loss"]
    supporting_value = result["loss"] + result["subgradient"] @ (point - predicted_cost)
    margins.append({"Draw": index + 1, "Shifted loss": shifted_loss, "Supporting value": supporting_value, "Margin": shifted_loss - supporting_value})
margin_table = pd.DataFrame(margins)
assert margin_table["Margin"].min() >= -1e-12
save_table(pd.DataFrame([
    {"Check": "Random comparison points", "Value": len(margin_table), "Pass": len(margin_table) == 100},
    {"Check": "Smallest convexity margin", "Value": margin_table["Margin"].min(), "Pass": margin_table["Margin"].min() >= -1e-12},
    {"Check": "Largest convexity margin", "Value": margin_table["Margin"].max(), "Pass": True},
]), "spo_subgradient_inequality_checks", "Finite supporting inequality checks for the selected subgradient.", "tab:chapter28-subgradient-checks")
margin_table.to_csv(FIGURE_DATA / "subgradient_margins.csv", index=False)
figure, axis = plt.subplots(figsize=(9.2, 5.2), constrained_layout=True)
axis.scatter(margin_table["Draw"], margin_table["Margin"], s=24, color=PALETTE["cyan"], alpha=0.82)
axis.axhline(0, color=PALETTE["rose"], linewidth=1.4)
axis.set_xlabel("Random comparison point")
axis.set_ylabel("Loss minus supporting value")
axis.set_title("The Selected Subgradient Passes the Supporting Inequality")
finish_axis(axis)
save_figure(figure, "subgradient_margins")

direction = np.array([1.0, -0.6])
grid = np.linspace(-2.2, 2.2, 181)
slice_rows = []
for value in grid:
    point = predicted_cost + value * direction
    item = spo_plus_loss(point, true_cost, vertices)
    slice_rows.append({"Position": value, "SPO+ loss": item["loss"], "Directional subgradient": float(item["subgradient"] @ direction), "Transformed vertex": item["transformed_vertex"]})
slice_table = pd.DataFrame(slice_rows)
slice_table.to_csv(FIGURE_DATA / "spo_plus_slice.csv", index=False)
figure, axes = plt.subplots(2, 1, figsize=(9.2, 7.2), sharex=True, constrained_layout=True)
axes[0].plot(slice_table["Position"], slice_table["SPO+ loss"], color=PALETTE["green"], linewidth=2.2)
axes[1].step(slice_table["Position"], slice_table["Directional subgradient"], where="mid", color=PALETTE["amber"], linewidth=2.0)
axes[0].set_ylabel("SPO+ loss")
axes[1].set_ylabel("Selected slope")
axes[1].set_xlabel("Position along a prediction line")
axes[0].set_title("SPO+ Is Convex and Piecewise Linear on the Finite Example")
for axis in axes:
    finish_axis(axis)
save_figure(figure, "spo_plus_slice")

,Component,RiskLabAI subgradient,Torch gradient,Absolute difference
0,1,-2,-2,0
1,2,2,2,0


,Check,Value,Pass
0,Random comparison points,100,Pass
1,Smallest convexity margin,-1.77636e-15,Pass
2,Largest convexity margin,2.81512,Pass


<Figure size 920x520 with 1 Axes>

<Figure size 920x720 with 2 Axes>

## Handling edge cases in decision loss

When $\widehat c=c$ and the observed best decision is unique, both the decision loss and SPO+ loss are zero. At a tie, the first supplied vertex is selected. These statements describe the declared finite oracle. They do not imply that the supplied vertices are a complete description of an unrelated optimization problem.

In [3]:
exact_result = spo_plus_loss(true_cost, true_cost, vertices)
tie_vertices = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]])
tie_result = spo_plus_loss(np.zeros(2), np.zeros(2), tie_vertices)
tie_checks = pd.DataFrame([
    {"Case": "Exact costs away from ties", "True vertex": exact_result["true_vertex"], "Transformed vertex": exact_result["transformed_vertex"], "Loss": exact_result["loss"], "Pass": np.isclose(exact_result["loss"], 0)},
    {"Case": "All costs tied", "True vertex": tie_result["true_vertex"], "Transformed vertex": tie_result["transformed_vertex"], "Loss": tie_result["loss"], "Pass": tie_result["true_vertex"] == tie_result["transformed_vertex"] == 0},
])
save_table(tie_checks, "spo_tie_and_exact_cost_checks", "Exact-cost and first-index tie checks.", "tab:chapter28-tie-checks", fit=True)

invalid_calls = [
    ("Cost dimensions differ", lambda: spo_plus_loss([1], [1, 2], [[1, 2]])),
    ("Vertex list is empty", lambda: spo_plus_loss([1], [2], [])),
    ("Nonfinite cost", lambda: spo_plus_loss([np.nan, 0], [1, 2], [[1, 0], [0, 1]])),
]
invalid_rows = []
for name, call in invalid_calls:
    rejected = False
    error_type = ""
    try:
        call()
    except ValueError as error:
        rejected = True
        error_type = type(error).__name__
    invalid_rows.append({"Case": name, "Rejected": rejected, "Error": error_type})
invalid_table = pd.DataFrame(invalid_rows)
assert invalid_table["Rejected"].all()
save_table(invalid_table, "spo_invalid_input_checks", "Explicit rejection checks for invalid oracle inputs.", "tab:chapter28-spo-invalid-inputs")

,Case,True vertex,Transformed vertex,Loss,Pass
0,Exact costs away from ties,2,2,0,Pass
1,All costs tied,0,0,0,Pass


,Case,Rejected,Error
0,Cost dimensions differ,Pass,ValueError
1,Vertex list is empty,Pass,ValueError
2,Nonfinite cost,Pass,ValueError


,Case,Rejected,Error
0,Cost dimensions differ,Pass,ValueError
1,Vertex list is empty,Pass,ValueError
2,Nonfinite cost,Pass,ValueError
